# Self-Refine — a multi-agent refinement loop (Reflexion-style)

[Reflexion](https://arxiv.org/abs/2303.11366) (Shinn et al.) and
[Self-Refine](https://arxiv.org/abs/2303.17651) (Madaan et al.) share one idea:
an answer gets better when a critic tells the producer *exactly* what it got
wrong and the producer tries again.

## Problem

How can we take any open-ended task — research a topic, fix a bug, write an
explainer — and make the output genuinely improve with each attempt, instead of
the model confidently repeating the same mistakes? A single LLM pass routinely
produces factual errors, misses constraints, writes vague sections, or makes
contradictory claims — and the model has no built-in mechanism to catch and fix
its own errors. The central challenge: **can we build a system that produces an
answer, finds everything wrong with it, feeds those findings back, and repeats
until the answer is genuinely solid — without the loop drifting into sycophancy
or runaway cost?**

## Description

We rebuild the Reflexion / Self-Refine loop as three named agents in a closed
loop using the Vidbyte SDK. A **Planner** decomposes the task into typed,
verifiable subgoals. A tool-using **Worker** executes the plan from scratch
each round, carrying only a verbal critique of its last attempt — it never
edits its prior draft. A **Debator** (devil's advocate) produces a structured
fault list whose length is the loop's deterministic stop signal. A small
**classifier** agent picks the worker's handoff-document shape once, up front.
The architecture is generic — swap the worker's tools and the system adapts
to any task domain.

## Notebook Table of Contents

| Section | What you'll build |
|---|---|
| **1. Environment & constants** | Provider setup, loop thresholds (`MAX_LOOP_ITERATIONS`, `DEBATOR_ITEMS`), worker budget caps |
| **2. System prompts** | Detailed prompts for all four agents — classifier, planner, worker, and debator at three strength levels |
| **3. Tools** | The worker's ground-truth toolbox — keyless Wikipedia `search` and `read_article` |
| **4. Typed contracts** | Pydantic schemas for `Plan`, `Critique`, and `HandoffChoice` — typed stage boundaries that make the stop rule deterministic |
| **5. Middleware** | Budget and runtime guards attached directly to the worker to cap cost per attempt |
| **6. Agents** | Constructing the classifier, planner, worker factory, and debator with the SDK |
| **7. The loop** | The `SelfRefineLoop` orchestrator — a Python class that wires plan → execute → critique → repeat |
| **8. Running the loop** | Multiple task examples: research, coding, comparison, and creative — each stresses the loop differently |
| **9. Next Steps** | Questions to guide your own improvements before you extend the system |

```text
                  +---- critique (typed fault list) ----+
                  v                                      |
  task --> PLANNER --plan--> WORKER --output+handoff--> DEBATOR
           output_schema     tools + handoff=            output_schema
           =Plan             (fresh attempt each round)  =Critique

  handoff shape picked ONCE by a CLASSIFIER -> Engineering | Research | Minimal
  stop when:  iterations == max_loop_iterations   OR   faults < debator_items
  final answer = latest worker output
```

**SDK primitives this notebook showcases:**
- `output_schema` + Pydantic — typed `Plan` and typed `Critique`; the critique's
  `len(items)` is the loop's deterministic stop signal (no bullet-counting).
- `handoff=` + prebuilt handoffs (`EngineeringHandoff` / `ResearchHandoff` /
  `MinimalHandoff`) — the worker emits a cold-start handoff doc describing what it did.
- `AgentInput(context_items=...)` — drops that handoff into the debator as a
  context primitive, with no string-stuffing.
- `.fork()` — spin off the planner and debator with empty history each round (the
  worker is rebuilt with its runtime-chosen handoff), so only the critique crosses rounds.
- `TokenBudgetMiddleware` / `CostBudgetMiddleware` / `RuntimeLimitMiddleware` —
  guards for a loop whose cost scales as roughly `3 x iterations`.

> **Before running:** copy `.env.example` to `.env` at the repo root and add your provider key.

## 1. Environment & constants

| Constant | What it controls |
|---|---|
| `PROVIDER` / `MODEL` | LLM for every agent. Swap to `anthropic` / `claude-opus-4-8` for a different model. |
| `MAX_LOOP_ITERATIONS` | Hard cap on plan -> execute -> critique rounds. The loop never exceeds this, no matter what. |
| `DEBATOR_ITEMS` | Stop when the debator finds **fewer than** this many faults. `1` = stop only at zero faults; `3` = stop at 0-2 faults. |
| `DEBATOR_STRENGTH` | Which debator system prompt to use. `"mild"` = constructive reviewer; `"moderate"` = rigorous devil's advocate (default); `"aggressive"` = uncompromising perfectionist. Each produces a different critique density. |
| `MAX_TOOL_ROUNDS` | How many search -> read cycles the worker may run per attempt. |
| `TOKEN_BUDGET` / `MAX_COST_PER_AGENT` / `COST_PER_MTOK` | Per-worker-attempt token and dollar caps (Section 5). `CostBudgetMiddleware` needs your model's blended $/1M-token rate to turn tokens into dollars. |

In [ ]:
%pip install -q vidbyte-sdk python-dotenv requests pydantic

import os
from dataclasses import dataclass, field

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

load_dotenv()
load_dotenv("../../.env")

PROVIDER = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL    = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")

MAX_LOOP_ITERATIONS = 3       # hard cap on plan -> execute -> critique rounds
DEBATOR_ITEMS       = 1       # stop when the debator finds FEWER than this many faults
DEBATOR_STRENGTH    = "moderate"  # "mild" | "moderate" | "aggressive" — controls critique density
MAX_TOOL_ROUNDS     = 6       # search -> read cycles the worker may run per attempt
TOKEN_BUDGET        = 25_000  # per-worker-attempt token cap (middleware)
MAX_COST_PER_AGENT  = 0.40    # per-worker-attempt dollar cap (middleware)
COST_PER_MTOK       = 4.0     # rough blended $ per 1M tokens for MODEL; set this for your model

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + overrides) in .env first."

## 2. System prompts

Four roles, four prompts — each modeled on the Vidbyte SDK master prompt
pattern (Identity → Goal → Checklist). Three are the named loop agents; the
fourth is a tiny **classifier** that decides the handoff shape once, up front.

Each prompt is detailed and prescriptive because in a multi-agent system,
the prompt is the agent's *operating manual*. Vague prompts produce vague
output, and vague output breaks the typed contracts that the loop depends on.

The **debator** comes in three strength levels (mild, moderate, aggressive) so
you can tune how much criticism the loop generates — aggressive catches more
but may burn iterations, mild converges faster but may miss subtle faults.

### 2a. Classifier

The classifier runs **once** before the loop starts. Its job is to read the
task and pick the right handoff-document shape — `engineering` for code tasks,
`research` for information-synthesis tasks, `minimal` for everything else. This
is how "the agent decides the handoff shape" becomes a concrete, deterministic
step instead of a hand-wavy instruction.

In [ ]:
CLASSIFIER_PROMPT = """# Identity
You are a task classifier that determines the correct handoff-document shape
for any task. You read the full task description and map it to exactly one
of three prebuilt handoff shapes.

# Goal
Your goal is to pick the handoff shape that best matches the nature of the
output the task expects. Your choice determines how the worker agent documents
its completed work for downstream agents. Choose based on the primary artifact
the task produces: code artifacts → engineering, information synthesis →
research, everything else → minimal.

# Checklist
- Read the full task description before choosing — do not guess from the first sentence
- engineering: the task produces or changes code, configs, tests, or a concrete build artifact
- research: the task gathers, evaluates, and synthesizes information into cited findings
- minimal: anything else — a short summary plus next steps is enough
- Return exactly the kind and a one-line reason — nothing else
- Do not overthink edge cases; pick the best fit and move on"""

### 2b. Planner

The planner turns the task into ordered, verifiable subgoals. On the first
round it sees only the task. On later rounds it also receives the latest
critique and restructures the plan so the next attempt cannot repeat those
faults. The planner gets **no tools** — planning is pure reasoning over text.

In [ ]:
PLANNER_PROMPT = """# Identity
You are a world-class Planner in a self-refinement loop. Your strategic
decomposition capability is unmatched — you analyze complex tasks with deep
logical depth, identify hidden dependencies and structural requirements, and
break work into concrete, independently verifiable subgoals. You serve as the
core orchestrator, defining clean boundaries and precise acceptance criteria
so downstream workers can execute without ambiguity.

# Goal
Your absolute goal is to turn any task into a short, ordered list of 3-6
subgoals that are jointly sufficient to complete the task. Every subgoal needs
a concrete, objective 'done_when' acceptance criterion — never vague labels
like "research the topic." When handed a CRITIQUE of a previous attempt, treat
it as the top priority: restructure the plan so the next attempt cannot repeat
those faults. Keep the plan tight and focused; do not add padding subgoals.

# Checklist
- Identify all functional and non-functional requirements embedded in the task
- Decompose the task into independent subgoals where possible to maximize clarity
- Define strict input-output contracts implicit in each subgoal
- Write concrete 'done_when' criteria that can be verified objectively
- If a CRITIQUE is provided, restructure the plan so no listed fault can recur
- Anticipate edge cases, boundary conditions, and typical failure modes for the task domain
- Keep subgoals self-contained to avoid context loss between execution steps
- Avoid redundant or circular subgoals — each must add real progress
- Adjust planning granularity based on task complexity: simpler tasks get fewer subgoals
- Ensure all budget, length, and format constraints are built into the subgoals"""

### 2c. Worker

The worker executes the plan with tools. It is told it is **starting fresh**
each round — it never sees its own previous output. The only history that
crosses the round boundary is the verbal critique checklist. This is the
Reflexion property: the worker re-attempts from scratch, guided only by the
plan and what the debator said went wrong last time.

In [ ]:
WORKER_PROMPT = """# Identity
You are a world-class Worker in a self-refinement loop. You execute assigned
tasks with meticulous rigor, grounding every factual claim in tool-verified
evidence and respecting every explicit constraint — length, format, scope,
style, and precision requirements. You never assert anything you have not
confirmed with a tool or verified against provided source material. You
produce complete, drop-in deliverables: no placeholders, no TODOs, no
hand-waving about "further research needed."

# Goal
Your absolute goal is to carry out the supplied task by following the supplied
plan. Use your tools to ground every factual claim — do not assert anything
you did not verify with a tool or source. Produce the best complete answer you
can and respect every explicit limit in the task (word count, output format,
scope boundaries). If handed a CRITIQUE checklist of what went wrong last
time, you are starting FRESH — do NOT assume any previous work exists — but
make sure your new answer repeats none of the listed faults.

# Checklist
- Follow the plan's subgoals in order; address each one explicitly
- Ground every factual or technical claim in tool output — no unsupported assertions
- Respect all task constraints: length limits, format requirements, scope rules
- Verify claims for internal consistency — check that no section contradicts another
- Cite specific sources or tool results for key factual claims
- If the task specifies output formatting (tables, sections, word count), match it exactly
- When given a CRITIQUE, actively avoid each listed fault in the new attempt
- Produce a complete answer — no placeholders, stubs, or references to missing research
- Structure output for readability: use clear sections, paragraphs, and lists
- Do not over-deliver beyond the task scope — extra sections dilute the requested output
- If a tool call fails or returns unexpected data, adapt but do not fabricate results"""

### 2d. Debator (three strength levels)

The debator is the devil's advocate. Its only job is to find faults and to
return an *empty* list when the work is genuinely solid. That empty list is
what stops the loop, so the prompt must discourage both sycophancy and padding.

We provide three strength levels so you can tune the loop's behavior:

| Strength | Behavior | Best for |
|---|---|---|
| `mild` | Constructive reviewer — flags clear errors and omissions, lets minor style pass. Converges fastest. | Tasks with objective correctness (math, structured data). |
| `moderate` | Rigorous devil's advocate — finds hidden assumptions, vague statements, and unsupported claims. Default. | General-purpose use; balances thoroughness with convergence speed. |
| `aggressive` | Uncompromising perfectionist — finds every imprecision, demands evidence for every claim. May never converge on subjective tasks. | High-stakes outputs where any error is costly; pushes the worker to its limit. |

The selected prompt is set by `DEBATOR_STRENGTH` at the top of the notebook.
Swap it and re-run to see how critique density changes.

In [ ]:
DEBATOR_PROMPTS = {
    "mild": """# Identity
You are a constructive Critic in a self-refinement loop — a reviewer who finds
genuine issues but maintains a helpful, improvement-oriented tone. You evaluate
work against its stated requirements and known facts, pointing out clear gaps
and inaccuracies without nitpicking. You err on the side of letting minor
style preferences pass if the substance is correct.

# Goal
Your goal is to evaluate the worker's output against the original task
requirements. Identify clear, substantiated faults: factual errors, missing
requirements, internal contradictions, and scope violations. Return one
critique item per distinct fault with a severity rating and a concrete fix
hint. If the output is genuinely solid — all requirements met, facts verified,
constraints respected — return an empty list. Be specific: name or quote the
exact part you object to. Do not invent faults just to fill the list.

# Checklist
- Read the task, the worker output, and any handoff document carefully
- Check every task requirement against the output — note anything missing
- Verify factual claims against sources cited in the output, not your own knowledge
- Flag unsupported assertions that lack tool evidence or citations
- Check length, format, and scope constraints explicitly stated in the task
- Ensure internal consistency: no section contradicts another
- Rate each fault as high (requirement missed, factual error), medium (vague, unsupported), or low (style, minor omission)
- Write fix hints that tell the worker exactly what to do differently
- Do not critique style choices unless they violate explicit task instructions
- If the output is solid across all checks, return an empty list — do not pad with weak objections""",

    "moderate": """# Identity
You are a rigorous Critic in a self-refinement loop — a relentless devil's
advocate who evaluates work with deep skepticism, looking for silent failures,
hidden assumptions, and boundary violations. You maintain a highly analytical,
fact-based stance without any emotional bias. Your feedback is direct,
objective, and entirely based on concrete requirements and logical constraints.
You serve as the system's internal quality gate, identifying weaknesses before
they can escape to the final answer.

# Goal
Your absolute goal is to evaluate the worker's output against the original
task, the supplied plan, and domain best practices. Find everything wrong:
unsupported or inaccurate claims, parts of the task it missed, vague or
unstructured sections, internal contradictions, limits it ignored, and
anything the handoff document overstates relative to what was actually
produced. Return one critique item per DISTINCT fault, each with a severity
and a concrete fix hint. If the answer is genuinely solid — all requirements
met, all claims grounded, all constraints respected — return an empty list.

# Checklist
- Evaluate the worker output line-by-line against the original task requirements
- Scan for silent logic errors, factual mistakes, or unvalidated assumptions
- Verify every claim is supported by tool output, citations, or clear reasoning
- Check that all task constraints (length, format, scope, word count) are strictly met
- Identify any forms of hallucination, extrapolation, or bias in the response
- Flag vague or unstructured sections that lack concrete detail
- Check for internal contradictions between different parts of the output
- Assess structural completeness — are all plan subgoals actually addressed?
- Compare the handoff document against the actual output — note any overstatements
- Check that sources are cited where the task requires grounding
- Rate severity: high (wrong facts, missed requirements), medium (unsupported claim, vagueness), low (minor clarity or organization)
- Write fix hints as direct, actionable instructions the worker can follow
- Reject any output containing plain placeholders, TODOs, or incomplete logic
- Do NOT be charitable, but do NOT invent faults to pad the list
- If the output is genuinely solid across all checks, return an empty list of items""",

    "aggressive": """# Identity
You are an uncompromising adversarial Critic in a self-refinement loop — a
relentless quality enforcer whose sole purpose is to find every flaw, no
matter how small. You are the system's last line of defense against sloppy
work. Mediocrity does not pass you. You interrogate every claim, demand
evidence for every assertion, and reject anything that falls short of complete
rigor and precision. Your bar is perfection; anything less deserves explicit
critique. You are feared by workers and respected by the system.

# Goal
Your absolute goal is to find and document every single problem in the
worker's output, down to the smallest imprecision. Attack it from every angle:
factual accuracy, logical coherence, completeness against requirements,
constraint compliance, structural clarity, citation quality, and tone
consistency. Return one critique item per distinct fault — even minor ones.
Do not soften your language. Every fault gets a severity rating and a
concrete fix hint. Only return an empty list if the output is genuinely
flawless by the strictest possible standard.

# Checklist
- Read the task, plan, and worker output with maximum skepticism — assume nothing
- Verify every factual claim independently — flag anything you cannot confirm
- Check every task requirement individually — flag ANY deviation, however small
- Hunt for imprecise language: "many", "often", "typically" without quantification
- Flag any claim stated as fact without an explicit source or tool verification
- Check word count, format, and structural requirements with zero tolerance
- Identify weak transitions, unclear antecedents, and ambiguous phrasing
- Verify that all numbers, dates, and technical terms are accurate and precise
- Check for missing nuance: does the answer oversimplify a complex topic?
- Flag any section where the handoff document claims more rigor than the output shows
- Assess whether the answer would satisfy a domain expert — not just a general reader
- Rate severity: high (any factual error, missed requirement), medium (unsupported claim, vague reasoning), low (word choice, minor structure)
- Write fix hints that are prescriptive and precise — no general advice
- Do not accept "good enough" — if you can find something to improve, include it
- Return an empty list ONLY if every possible dimension of quality is met""",
}

DEBATOR_PROMPT = DEBATOR_PROMPTS[DEBATOR_STRENGTH]

## 3. Tools

The worker's only window into reality. It gets two keyless tools over
Wikipedia's public API — `search` and `read_article` — so the notebook runs
without a search-API key. These are the "whatever tools" slot of the harness:
swap them for filesystem tools, a code runner, or a SQL client and the loop
structure is unchanged.

**The planner and debator get no tools** — planning and critiquing are pure
reasoning over text. Only the worker interacts with the outside world.

In [ ]:
import re
from vidbyte import tool

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS  = {"User-Agent": "vidbyte-cookbook-self-refine/0.1"}


@tool
def search(query: str) -> list[dict]:
    """Search Wikipedia for articles matching the query. Returns up to 5 results with
    title and a short snippet. Use read_article to read one in full."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "list": "search", "srsearch": query,
        "srlimit": 5, "format": "json",
    })
    resp.raise_for_status()
    return [
        {"title": h["title"], "snippet": re.sub(r"<[^>]+>", "", h["snippet"])}
        for h in resp.json()["query"]["search"]
    ]


@tool
def read_article(title: str) -> str:
    """Read the plain-text extract of one Wikipedia article by its exact title (truncated
    to 6 000 chars). Use the exact title returned by search()."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "prop": "extracts", "explaintext": 1,
        "titles": title, "format": "json", "redirects": 1,
    })
    resp.raise_for_status()
    pages = resp.json()["query"]["pages"]
    extract = next(iter(pages.values())).get("extract", "")
    return extract[:6_000] if extract else f"No article found for '{title}'."


WORKER_TOOLS = [search, read_article]

**Pause and think — tools:**

1. The worker's only tools are `search` and `read_article`. If you pointed this
   loop at a **code-fix** task, which two tools would you add to the worker, and
   which single read-only tool would you give the **planner** so it stops planning
   blind to the codebase?

2. Why does the debator get *no* tools? What failure mode would a web-searching
   debator introduce that a text-only debator avoids?

3. The worker re-attempts *from scratch* each round. Name one task where feeding
   the worker its previous draft would clearly help, and one where it would make
   the worker stubbornly defend a bad first draft. Which kind is the demo task below?

4. If you wanted the worker to also modify files (not just read), what extra SDK
   permission would you need to grant, and where would you attach it?

> **More tools you could add:** The Vidbyte SDK ships with a large catalog of
> prebuilt tools that drop into `WORKER_TOOLS` without writing code. For a
> code-fix task you could use `ReadTextTool`, `WriteTextTool`, `ListDirTool`
> (from `vidbyte.tools.filesystem`), `GlobTool`, `GrepTool` (from
> `vidbyte.tools.builtins.code_search`), and `PatchTool` (from
> `vidbyte.tools.builtins.editing`). For agent memory you could use
> `Mem0AddMemoryTool` or `ZepSearchMemoryTool` (from
> `vidbyte.tools.builtins.memory`). See `vidbyte-sdk/skills/usage/available_tools.md`
> for the full catalog — the loop does not care which tools you give the worker.

## 4. Typed contracts

The Pydantic models below are the typed boundaries every agent speaks. They are
passed as `output_schema` to `BaseAgent`; the SDK returns the validated instance
in `reply.metadata["structured"]`.

Two are load-bearing: `Plan.subgoals` is what the worker executes, and
`Critique.items` is what the loop *counts* to decide whether to stop. Typing the
critique is why the stop rule is a clean `len(items) < DEBATOR_ITEMS` instead of
a fragile bullet-parser.

In [ ]:
# ---- Typed stage contracts (output_schema) --------------------------------
class Subgoal(BaseModel):
    goal: str = Field(description="One concrete subgoal toward completing the task.")
    done_when: str = Field(description="Objective acceptance criterion for this subgoal.")


class Plan(BaseModel):
    summary: str = Field(description="One-sentence restatement of what the task must achieve.")
    subgoals: list[Subgoal] = Field(description="3-6 ordered, jointly sufficient subgoals.")


class CritiqueItem(BaseModel):
    issue: str = Field(description="Specifically what the worker got wrong.")
    severity: str = Field(description='"high", "medium", or "low".')
    fix_hint: str = Field(description="What a corrected attempt should do instead.")


class Critique(BaseModel):
    items: list[CritiqueItem] = Field(description="One entry per distinct fault; empty means no faults found.")
    verdict: str = Field(description="One-line overall assessment of the output.")


class HandoffChoice(BaseModel):
    kind: str = Field(description='Exactly one of "engineering", "research", "minimal".')
    reason: str = Field(description="One line on why this shape fits the task.")


def get_structured(reply, schema):
    """Pull the validated output_schema payload out of a reply, or raise if it failed to parse."""
    payload = reply.metadata.get("structured")
    if payload is None:
        raise RuntimeError(f"{schema.__name__} validation failed: {reply.content[:200]}")
    return payload if isinstance(payload, schema) else schema.model_validate(payload)


def render_plan(plan):
    """Render a Plan as a numbered checklist string for an agent prompt."""
    return "\n".join(f"{i}. {s.goal} (done when: {s.done_when})" for i, s in enumerate(plan.subgoals, 1))


def render_faults(critique):
    """Render a Critique as a severity-tagged bulleted list string."""
    if not critique.items:
        return "- (none)"
    return "\n".join(f"- [{c.severity}] {c.issue}  ->  fix: {c.fix_hint}" for c in critique.items)

**Pause and think — typed contracts:**

1. The stop rule is `len(items) < DEBATOR_ITEMS`. Why is a typed `Critique.items`
   list more reliable than counting bullet points in a free-text string? What
   specific failure would a regex-based bullet counter have?

2. `Plan.subgoals` each have a `done_when` field. If the worker completes all
   subgoals but the debator finds a fundamental factual error, should the planner
   add a new subgoal or rewrite an existing one — and why?

3. `HandoffChoice` is classified *once* before the loop. What would happen if you
   re-classified every round instead — would the result ever change, and would
   the extra model call be worth it?

4. The `CritiqueItem.severity` field constrains the debator to `high`/`medium`/`low`.
   How would you use severity to build a smarter stop rule — for example, "stop
   when there are no high-severity faults remaining, even if low-severity faults
   persist"?

## 5. Middleware

Middleware is deterministic runtime policy that the model never sees. A
refinement loop is exactly the place you want it: the worker runs its own tool
loop, and the *outer* loop multiplies every cost by the number of iterations.
We attach budget + runtime guards to the worker (the only agent that loops over
tools):

| Middleware | Role in the refinement loop |
|---|---|
| `TokenBudgetMiddleware` | Hard token cap per worker attempt — one expensive attempt can't blow the whole run. |
| `CostBudgetMiddleware` | Dollar cap per worker attempt — converts token usage to dollars via `cost_per_million_tokens`. |
| `RuntimeLimitMiddleware` | Wall-clock + model-call cap — a worker stuck in a search loop can't hang the round. |

Each worker attempt gets a **fresh** set of these guards (new instances), so
one round's budget never bleeds into the next. The middleware list is rendered
directly in the agent constructor — no separate helper function.

In [ ]:
from vidbyte.middleware import (
    CostBudgetMiddleware,
    RuntimeLimitMiddleware,
    TokenBudgetMiddleware,
)

> **More middleware you could add:** The Vidbyte SDK ships with 13+ built-in
> middlewares beyond the three used here. For a production refinement loop you
> might add `LoopDetectionMiddleware` (detects when the worker gets stuck in a
> tool-calling loop), `ModelRetryMiddleware` (handles transient provider errors
> with backoff), `AuditLogMiddleware` (structured logging of every lifecycle
> event for debugging), and `ToolPolicyMiddleware` (whitelist exactly which
> tools each agent may call). For context-window management you could use
> `MessageHistoryCompactionMiddleware` or `SummaryCompactionMiddleware` to keep
> the worker's growing tool-output history from blowing the context budget.
> See `vidbyte-sdk/skills/vidbyte-sdk/middleware.md` for the full catalog.

## 6. Agents

Four agents constructed with the Vidbyte SDK:

- **classifier** — `output_schema=HandoffChoice`, run once before the loop.
- **planner** — `output_schema=Plan`, no tools. **Forked** each round for fresh history.
- **make_worker(...)** — a *factory*, not a singleton. The handoff shape isn't
  known until the classifier runs, and `fork()` can't override an agent's
  `handoff=` (a fork always inherits its parent's handoff spec). So we build a
  fresh worker each round with the chosen shape — which also gives every attempt
  empty history. The middleware list is constructed **directly inline** in each
  call to `make_worker`, so every round gets fresh budget guards.
- **debator** — `output_schema=Critique`, no tools; **forked** each round and
  handed the handoff via `context_items`.

> **Why fresh agents every round?** Reusing one long-lived agent would let its old
> message history leak into the next attempt — defeating the "start fresh" property.
> A fresh agent means the critique is the *only* thing that crosses the round boundary.

In [ ]:
from vidbyte import BaseAgent, AgentInput, EngineeringHandoff, ResearchHandoff, MinimalHandoff

HANDOFF_BY_KIND = {
    "engineering": EngineeringHandoff,
    "research": ResearchHandoff,
    "minimal": MinimalHandoff,
}

classifier = BaseAgent(
    name="handoff-classifier", system_prompt=CLASSIFIER_PROMPT,
    provider=PROVIDER, model_name=MODEL, output_schema=HandoffChoice,
)

planner = BaseAgent(
    name="planner", system_prompt=PLANNER_PROMPT,
    provider=PROVIDER, model_name=MODEL, output_schema=Plan,
)

def make_worker(handoff_spec, i):
    """Build a fresh worker with the chosen handoff shape for round i.
    Middleware is constructed inline so every attempt gets its own fresh guards."""
    return BaseAgent(
        name=f"worker-{i}", system_prompt=WORKER_PROMPT,
        provider=PROVIDER, model_name=MODEL, tools=WORKER_TOOLS,
        middleware=[
            TokenBudgetMiddleware(max_tokens=TOKEN_BUDGET),
            CostBudgetMiddleware(max_spend_usd=MAX_COST_PER_AGENT, cost_per_million_tokens=COST_PER_MTOK),
            RuntimeLimitMiddleware(max_model_calls=15, max_elapsed_seconds=120.0),
        ],
        max_tool_rounds=MAX_TOOL_ROUNDS,
        handoff=handoff_spec,
    )


debator = BaseAgent(
    name="debator", system_prompt=DEBATOR_PROMPT,
    provider=PROVIDER, model_name=MODEL, output_schema=Critique,
)

## 7. The loop

The orchestrator is a plain Python class, **not** a `SequentialPipeline`.
Pipelines move only strings between stages and can't loop or carry typed
objects — but this loop passes a `Plan` object forward, a `Handoff` object
sideways into the debator, and a `Critique` *backward* to the next planner.
A class makes those edges explicit.

Two design choices worth pausing on:
1. **A fresh agent every round.** Planner and debator are *forked* (empty
   history); the worker is *rebuilt*. Either way, reusing one long-lived agent
   would let its old message history leak into the next attempt — defeating the
   "start fresh" property.
2. **Handoff chosen once.** The task's nature doesn't change between rounds, so
   we classify the shape a single time and reuse it; re-classifying each round
   would cost a model call for no new signal.

In [ ]:
@dataclass
class RefineResult:
    final_output: str        # the latest iteration's worker output (the returned answer)
    final_plan: Plan         # the plan that produced it
    stop_reason: str         # "converged" | "max_iterations"
    iterations_run: int
    history: list = field(default_factory=list)


class SelfRefineLoop:
    """Runs plan -> execute -> critique on a task until the debator runs out of faults or the cap."""

    def __init__(self, task, *, max_loop_iterations=MAX_LOOP_ITERATIONS, debator_items=DEBATOR_ITEMS):
        self.task = task
        self.max_loop_iterations = max_loop_iterations
        self.debator_items = debator_items
        self.handoff_spec = None
        self.history: list[dict] = []

    def run(self) -> RefineResult:
        """Pick the handoff shape once, then loop until a stop condition fires; return the latest output."""
        self.handoff_spec = self._choose_handoff()
        critique, plan, output = None, None, ""
        for i in range(1, self.max_loop_iterations + 1):
            plan = self._plan(critique, i)
            output, handoff_doc = self._execute(plan, critique, i)
            critique = self._critique(output, handoff_doc, i)
            self._record(i, plan, output, handoff_doc, critique)
            stop, reason = self._should_stop(critique, i)
            print(f"  iteration {i}: {len(critique.items)} fault(s) -> {reason or 'continue'}")
            if stop:
                return RefineResult(output, plan, reason, i, self.history)
        return RefineResult(output, plan, "max_iterations", self.max_loop_iterations, self.history)

    def _choose_handoff(self):
        """Classify the task once into a prebuilt handoff shape and reuse it every iteration."""
        choice = get_structured(classifier.run(self.task), HandoffChoice)
        spec_cls = HANDOFF_BY_KIND.get(choice.kind, MinimalHandoff)
        print(f"Handoff shape: {choice.kind} - {choice.reason}")
        return spec_cls()

    def _plan(self, critique, i) -> Plan:
        """Produce a fresh plan; on later rounds the latest critique is the priority input."""
        prompt = f"TASK:\n{self.task}"
        if critique is not None:
            prompt += "\n\nCRITIQUE OF THE PREVIOUS ATTEMPT (restructure to fix these):\n" + render_faults(critique)
        return get_structured(planner.fork(name=f"planner-{i}").run(prompt), Plan)

    def _execute(self, plan, critique, i):
        """Run a fresh worker against the plan (+ critique checklist) and return its output and handoff."""
        prompt = f"TASK:\n{self.task}\n\nPLAN:\n{render_plan(plan)}"
        if critique is not None:
            prompt += "\n\nDO NOT REPEAT THESE FAULTS FROM THE LAST ATTEMPT:\n" + render_faults(critique)
        worker = make_worker(self.handoff_spec, i)
        reply = worker.run(prompt)
        handoff_doc = reply.metadata.get("handoff") or worker.last_handoff
        if handoff_doc is None:
            print("  (note: handoff generation returned nothing; debator will review output text only)")
        return reply.content, handoff_doc

    def _critique(self, output, handoff_doc, i) -> Critique:
        """Have a fresh debator attack the output; pass the handoff as a context item when present."""
        prompt = f"TASK:\n{self.task}\n\nWORKER OUTPUT TO REVIEW:\n{output}"
        agent_input = AgentInput(prompt, context_items=(handoff_doc,)) if handoff_doc else prompt
        return get_structured(debator.fork(name=f"debator-{i}").run(agent_input), Critique)

    def _should_stop(self, critique, iteration):
        """Stop when the debator finds fewer than debator_items faults, or the iteration cap is hit."""
        if len(critique.items) < self.debator_items:
            return True, "converged"
        if iteration >= self.max_loop_iterations:
            return True, "max_iterations"
        return False, ""

    def _record(self, iteration, plan, output, handoff_doc, critique):
        """Append one iteration's artifacts to the in-memory run log."""
        self.history.append({
            "iteration": iteration, "plan": plan, "output": output,
            "handoff": handoff_doc, "critique": critique, "n_faults": len(critique.items),
        })

## 8. Running the loop

Instantiate `SelfRefineLoop` with a task and call `run()`. Watch the fault count
per round: a healthy run trends toward zero. The returned `RefineResult` carries
the latest answer plus the full per-round history.

Below are four example tasks — each stresses the loop differently:

| Task | Type | What it tests |
|---|---|---|
| **Hindenburg explainer** | Research (factual) | Accuracy of historical claims, word-count constraint, source grounding |
| **TCP vs UDP** | Research (comparison) | False symmetry detection, technical precision, concrete examples |
| **Bug fix ticket** | Coding (requires swapping tools) | Reproduce → fix → verify loop, code-level precision |
| **Product taglines** | Creative (subjective) | Early termination problem — the debator can always nitpick |

Run the first task below, then edit `TASK` to try the others. For the coding
task you would swap `WORKER_TOOLS` for filesystem tools — the loop structure
stays identical.

In [ ]:
# ── Research task: factual explainer with word-count constraint ──
TASK_RESEARCH = (
    "Write a tight, accurate 200-word explainer answering: why did the Hindenburg "
    "disaster effectively end the passenger-airship era? Ground every claim in sources "
    "you actually read, and stay at or under 200 words."
)

# ── Comparison task: technical topic with concrete examples ──
TASK_COMPARISON = (
    "Explain the practical differences between TCP and UDP for a backend engineer, "
    "in under 250 words, with a concrete real-world example use case for each. "
    "Do not just list features — explain the tradeoffs an engineer actually faces."
)

# ── Coding task: bug fix ticket (swap WORKER_TOOLS for filesystem tools first) ──
TASK_CODING = (
    "TICKET-1042: The apply_discount function treats a 0-100 percent as a 0-1 "
    "fraction. A 10% discount on $100 should give $90, not -$900. Fix it and "
    "make the test suite pass. Acceptance criteria: python -m unittest passes "
    "with exit code 0."
)

# ── Creative task: subjective (watch this tend to hit the iteration cap) ──
TASK_CREATIVE = (
    "Write three punchy one-line taglines for a note-taking app for researchers. "
    "Each tagline must be under 12 words and convey a different angle: speed, "
    "insight, and collaboration."
)

TASK = TASK_RESEARCH  # change this to try different task types

print("Running self-refinement loop...\n")
loop = SelfRefineLoop(TASK, max_loop_iterations=MAX_LOOP_ITERATIONS, debator_items=DEBATOR_ITEMS)
result = loop.run()

print(f"\nStopped after {result.iterations_run} iteration(s): {result.stop_reason}")
print("Faults per round:", [h["n_faults"] for h in result.history])

print("\nWhat the debator caught in round 1:")
print(render_faults(result.history[0]["critique"]))

print("\n" + "=" * 64 + "\nFINAL ANSWER (latest iteration)\n" + "=" * 64)
display(Markdown(result.final_output))

## 9. Next Steps

Instead of a static example output, here are five questions to guide your own
extensions. Pick one, implement it, and re-run the notebook.

1. **Tune the debator.** Set `DEBATOR_STRENGTH = "aggressive"` and re-run the
   creative tagline task. Does the loop converge, or does it burn all iterations?
   Now set `DEBATOR_ITEMS = 2` — does a small tolerance for minor faults let
   the aggressive debator still converge? What does this tell you about how the
   two settings interact?

2. **Add a second stop signal.** The loop stops on `len(items) < DEBATOR_ITEMS`,
   which is purely the debator's judgment. What if you added a deterministic
   check — for example, a word-count assert that bypasses the debator — or a
   diff against the previous round's output that stops when the answers barely
   change? Where in `_should_stop` would you add it, and what failure mode does
   it protect against?

3. **Give the worker real SDK tools.** Swap `WORKER_TOOLS` for the Vidbyte SDK's
   built-in filesystem and code-search tools (`ReadTextTool`, `GlobTool`, `GrepTool`,
   `PatchTool` — see the note in Section 3 for import paths). Point the loop at
   the coding task. Does the debator's critique change when the worker has
   different tools?

4. **Return the best round, not the latest.** When `stop_reason` is
   `"max_iterations"`, the returned answer is the last attempt — which might
   have *more* faults than an earlier round. Write a one-liner that returns
   `min(result.history, key=lambda h: h["n_faults"])` instead. Does this
   meaningfully change the output on the creative task?

5. **Accumulate reflection memory.** Currently only the *latest* critique is
   fed back. Real Reflexion keeps *all* past critiques in the worker's memory.
   Modify `SelfRefineLoop._execute` to include all previous critiques (not just
   the most recent) in the worker's prompt. What happens to convergence speed?
   What happens to token cost as the critique history grows?